In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import scipy.io as sio
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
import os
import random
from random import shuffle
import scipy.ndimage
from tensorflow.keras import regularizers
from keras.utils import to_categorical
from sklearn.model_selection import StratifiedKFold
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.metrics import classification_report, confusion_matrix
import pickle
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Dropout, Input, Concatenate
from tensorflow.keras.layers import BatchNormalization, Activation, GlobalAveragePooling2D, Multiply, Add, Reshape
import time
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
from tensorflow.keras import backend as K

In [ ]:
#this function is used to load all data i.e complete labelled samples
def loadData(path, filename):
    ''' x contains the features in image dimension and labels contains the actual gt'''
    data_path = os.path.join(os.getcwd(), path)
    data = sio.loadmat(os.path.join(data_path, filename))['final_6d_svd_data']
    y = sio.loadmat(os.path.join(data_path, 'ground_truth_flevoland.mat'))['label']
    x = data
    labels = y
    return x, labels


# filename = 'T3_SVD'
filename = 'final_6d_svd'

# data_path = os.path.join(os.getcwd(),"/kaggle/input/datasets/birchetry/sanfran/")
# data_path = os.path.join(os.getcwd(),"/kaggle/input/datasets/birchetry/rs2/")
data_path = os.path.join(os.getcwd(),"/kaggle/input/datasets/birchetry/svd6dflevo/")

X, labels = loadData(data_path, filename)
X.shape

In [ ]:
def normalize_data(data):
    # Convert to float32 for safe computation
    data = data.astype(np.float32)
    # Initialize output array
    normalized_cube = np.zeros_like(data, dtype=np.float32)
    # Loop over channels (bands)
    for band_idx in range(data.shape[2]):
        band = data[:, :, band_idx]
        min_val = np.min(band)
        max_val = np.max(band)
        if max_val > min_val:
            normalized_cube[:, :, band_idx] = (band - min_val) / (max_val - min_val)
        else:
            # If band is constant
            normalized_cube[:, :, band_idx] = 0.0
   
    return normalized_cube

In [ ]:
X = normalize_data(X)
X.shape

In [ ]:
def padWithZeros(X, margin=2):
    newX = np.zeros((X.shape[0] + 2 * margin, X.shape[1] + 2* margin, X.shape[2]))
    x_offset = margin
    y_offset = margin
    newX[x_offset:X.shape[0] + x_offset, y_offset:X.shape[1] + y_offset, :] = X
    return newX


def createPatches(X, y, windowSize=5, removeZeroLabels = True):
    margin = int((windowSize - 1) / 2)
    zeroPaddedX = padWithZeros(X, margin=margin)
    # split patches
    patchesData = np.zeros((X.shape[0] * X.shape[1], windowSize, windowSize, X.shape[2]))
    patchesLabels = np.zeros((X.shape[0] * X.shape[1]))
    patchIndex = 0
    for r in range(margin, zeroPaddedX.shape[0] - margin):
        for c in range(margin, zeroPaddedX.shape[1] - margin):
            patch = zeroPaddedX[r - margin:r + margin + 1, c - margin:c + margin + 1]
            patchesData[patchIndex, :, :, :] = patch
            patchesLabels[patchIndex] = y[r-margin, c-margin]
            patchIndex = patchIndex + 1
    if removeZeroLabels:
        patchesData = patchesData[patchesLabels>0,:,:,:]
        patchesLabels = patchesLabels[patchesLabels>0]
        patchesLabels -= 1
    return patchesData, patchesLabels

windowSize = 7
XPatches, yPatches = createPatches(X, labels, windowSize=windowSize)

XPatches.shape, yPatches.shape

In [ ]:
def reports(model, X_test, y_test):
    Y_pred = model.predict(X_test)
    y_pred = np.argmax(Y_pred, axis=1)

# #     ## target_names for Flevo
    target_names = ['a', 'b', 'c', 'd'
            ,'e', 'f', 'g',
                'h', 'i', 'j', 'k',  'l', 'm', 'n', 'o']

    ## target_names for SF
    # target_names = ['a', 'b', 'c', 'd', 'e']

    #
    classification = classification_report(np.argmax(y_test, axis=1), y_pred, target_names=target_names)
    confusion = confusion_matrix(np.argmax(y_test, axis=1), y_pred)
    score = model.evaluate(X_test, y_test, batch_size=256)
    print(score)

    Test_Loss =  score[0]*100
    Test_accuracy = score[1]*100

    return classification, confusion, Test_Loss, Test_accuracy

def generate_reports(XPatches, yPatches, model, filename, index):
    #creation of test patches to include all the samples
    Xp1 = XPatches ## FOR SVD
    # Reshape Xpatches2
    X_test = Xp1
    y_test = yPatches

    #convert the y labels to categorical
    y_test = to_categorical(y_test)

    classification, confusion, Test_loss, Test_accuracy = reports(model,X_test,y_test)

    # Calculate class-wise accuracy
    class_wise_acc = np.diag(confusion) / np.sum(confusion, axis = 1)

    # Calculate overall accuracy
    overall_acc = np.sum(np.diag(confusion)) / np.sum(confusion)

    # Calculate average accuracy
    average_acc = np.mean(class_wise_acc)

    # Calculate chance agreement
    chance_agreement = np.sum(np.sum(confusion, axis=1) * np.sum(confusion, axis=0)) / (np.sum(confusion) ** 2)

    # Calculate kappa
    kappa = (overall_acc - chance_agreement) / (1 - chance_agreement)

    print("Class-wise accuracy:", class_wise_acc)
    print("Overall accuracy:", overall_acc)
    print("Average accuracy:", average_acc)
    print("Kappa:", kappa)

    classification = str(classification)
    confusion = str(confusion)
    file_name = f"Accy of {filename}-{index}.txt"
    with open(file_name, 'w') as x_file:
        x_file.write('{} Test loss (%)'.format(Test_loss))
        x_file.write('\n')
        x_file.write('{} Test accuracy (%)'.format(Test_accuracy))
        x_file.write('\n')
        x_file.write('\n')
        x_file.write('{}'.format(classification))
        x_file.write('\n')
        x_file.write('{}'.format(confusion))
        x_file.write('\n')
        x_file.write('\n')
        x_file.write("Class-wise accuracy:{}".format(class_wise_acc))
        x_file.write('\n')
        x_file.write("Overall accuracy:{}".format(overall_acc))
        x_file.write('\n')
        x_file.write("Average accuracy:{}".format(average_acc))
        x_file.write('\n')
        x_file.write("Kappa:{}".format(kappa))

In [ ]:
def splitTrainTestSet(X, y, testRatio=0.9):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=testRatio, random_state=345,
                                                        stratify=y)
    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = splitTrainTestSet(XPatches, yPatches)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:

## Reshape of Train set
X_train1 = X_train
# X_train1.shape
X_train1.shape
X_test1 = X_test
# X_train1.shape
X_test1.shape

y_train = to_categorical(y_train)

# Define the input shape
input_shape1 = X_train1[0].shape
print("input_shape1: ",input_shape1)
# input_shape2 = X_train2[0].shape
# print("input_shape2: ",input_shape2)

In [ ]:
# SSPA MODULE
def sspa_block(input_tensor, ratio=2): 
    channels = K.int_shape(input_tensor)[-1]
    se_shape = (1, 1, channels)
    se = GlobalAveragePooling2D()(input_tensor)
    se = Reshape(se_shape)(se)
    se = Dense(
        channels // ratio, activation="relu", use_bias=False
    )(se)
    se = Dense(channels, activation="sigmoid", use_bias=False)(se)
    return Multiply()([input_tensor, se])

# PSSA MODULE
def pssa_block(input_tensor):
    se = Conv2D(
        1, (1, 1), activation="sigmoid", use_bias=False, kernel_regularizer=regularizers.l2(0.0001), kernel_initializer="he_normal"
    )(input_tensor)
    return Multiply()([input_tensor, se])

# SPFF BLOCK
def spff_block(input_tensor, ratio=2):
    spectral_se = sspa_block(input_tensor, ratio)
    spatial_se = pssa_block(input_tensor)
    return Add()([spectral_se, spatial_se])


def res_block_darn(input_tensor, filters, kernel_size, ratio=2):
    
    f1, f2, f3 = filters
    x = Conv2D(f1, (1, 1), kernel_regularizer=regularizers.l2(0.0001), kernel_initializer="he_normal")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(f2, kernel_size, padding="same", kernel_regularizer=regularizers.l2(0.0001), kernel_initializer="he_normal")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(f3, (1, 1), kernel_initializer="he_normal")(x)
 #   x = BatchNormalization()(x)

    x = spff_block(x, ratio)

    shortcut = Conv2D(f3, (1, 1), kernel_regularizer=regularizers.l2(0.0001), kernel_initializer="he_normal")(input_tensor)
#    shortcut = BatchNormalization()(shortcut)
    
    # Add shortcut to the main path
    x = Add()([x, shortcut])
 #   x = Activation("relu")(x)

    return x

def SVD_DARN(input_shape, num_classes, num_res_blocks=4):
   
    input_layer = Input(shape=input_shape)
    x = Conv2D(
        128,
        (1, 1),
        padding="same",
        kernel_regularizer=regularizers.l2(0.0001),
        kernel_initializer="he_normal",
    )(input_layer)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    # Stack of DARN Residual Blocks
    for _ in range(num_res_blocks):
        x = res_block_darn(x, filters=(32, 32, 128), kernel_size=(3, 3))

    # Global Pooling and Classification Head
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)  # Add dropout here (0.5 = 50% drop rate)
    output_layer = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=input_layer, outputs=output_layer, name="SVD_DARN")
   
    return model

svd_model = SVD_DARN(input_shape=(7,7,30),
                              num_classes=15)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
svd_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
svd_model.summary()

start = time.time()
svd_model.fit(X_train1, y_train, batch_size=64, epochs=100)
print("Total Training time: ", time.time() - start, "seconds")

In [ ]:
generate_reports(XPatches, yPatches, svd_model, filename, 1)